In [2]:
import pandas as pd
import lyricsgenius
import time

In [2]:
# Load your data
df_with_lyrics = pd.read_csv("data/features-annotated-artistless.csv")

# Get the first artist name
df_with_lyrics["artist"] = df_with_lyrics["Artists"].apply(
    lambda x: x.split(",")[0].strip() if pd.notnull(x) else x
)

In [3]:
# Genius setup
genius = lyricsgenius.Genius("llKa2e1wpiARUC2cKGw6ollI-o99YkqvE60E2KBXBconkOPxudklxUFi_qT7Pz2w",
                             skip_non_songs=True,
                             remove_section_headers=True,
                             timeout=10)

In [4]:
# Function to safely get lyrics
def get_lyrics(artist, title):
    try:
        song = genius.search_song(title, artist)
        if song and song.lyrics:
            return song.lyrics
    except Exception as e:
        print(f"Error fetching {artist} - {title}: {e}")
    return None

In [ ]:
# Apply function with rate limiting
lyrics_list = []
for idx, row in df_with_lyrics.iterrows():
    lyrics_list.append(get_lyrics(row["artist"], row["Title"]))
    time.sleep(2)  # avoid API rate limits

df_with_lyrics["lyrics"] = lyrics_list

# Save the result
df_with_lyrics.to_csv("data/features-annotated-artistless-with-lyrics.csv", index=False)
print("Lyrics added and file saved.")

Lyrics added and file saved.


In [3]:
import os
import psycopg2
from psycopg2.extras import execute_batch
from lingua import LanguageDetectorBuilder
from tqdm import tqdm

In [61]:
input_csv = "data/features-annotated-artistless-with-lyrics.csv"
output_csv = "data/features-annotated-artistless-with-lyrics-and-languages.csv"

df = pd.read_csv(input_csv)

# Replace NaN with empty string to avoid errors
lyrics_texts = df["lyrics"].fillna("").tolist()

In [62]:
detector = (
    LanguageDetectorBuilder
        .from_all_languages()
        .with_preloaded_language_models()
        .build()
)

In [63]:
print("Detecting languages (multi-threaded)...")

results = detector.detect_multiple_languages_in_parallel_of(lyrics_texts)

Detecting languages (multi-threaded)...


In [64]:
print(results[0])

[DetectionResult(start_index=0, end_index=149, word_count=33, language=Language.DUTCH), DetectionResult(start_index=149, end_index=200, word_count=10, language=Language.SPANISH), DetectionResult(start_index=200, end_index=278, word_count=16, language=Language.SOMALI), DetectionResult(start_index=278, end_index=655, word_count=82, language=Language.SPANISH), DetectionResult(start_index=655, end_index=695, word_count=8, language=Language.SOMALI), DetectionResult(start_index=695, end_index=806, word_count=25, language=Language.DUTCH), DetectionResult(start_index=806, end_index=852, word_count=9, language=Language.SPANISH), DetectionResult(start_index=852, end_index=931, word_count=14, language=Language.DUTCH), DetectionResult(start_index=931, end_index=947, word_count=2, language=Language.PORTUGUESE), DetectionResult(start_index=947, end_index=1212, word_count=43, language=Language.SPANISH), DetectionResult(start_index=1212, end_index=1230, word_count=2, language=Language.ROMANIAN), Detec

In [65]:
confidence_results = detector.compute_language_confidence_values_in_parallel(lyrics_texts)

In [48]:
print(confidence_results[0])

[ConfidenceValue(language=Language.SPANISH, value=1), ConfidenceValue(language=Language.AFRIKAANS, value=0), ConfidenceValue(language=Language.ALBANIAN, value=0), ConfidenceValue(language=Language.ARABIC, value=0), ConfidenceValue(language=Language.ARMENIAN, value=0), ConfidenceValue(language=Language.AZERBAIJANI, value=0), ConfidenceValue(language=Language.BASQUE, value=0), ConfidenceValue(language=Language.BELARUSIAN, value=0), ConfidenceValue(language=Language.BENGALI, value=0), ConfidenceValue(language=Language.BOKMAL, value=0), ConfidenceValue(language=Language.BOSNIAN, value=0), ConfidenceValue(language=Language.BULGARIAN, value=0), ConfidenceValue(language=Language.CATALAN, value=0), ConfidenceValue(language=Language.CHINESE, value=0), ConfidenceValue(language=Language.CROATIAN, value=0), ConfidenceValue(language=Language.CZECH, value=0), ConfidenceValue(language=Language.DANISH, value=0), ConfidenceValue(language=Language.DUTCH, value=0), ConfidenceValue(language=Language.ENGLI

In [ ]:
detected_languages = []
num_languages = []
language_confidences = []

for conf_values, text, detections in tqdm(zip(confidence_results, lyrics_texts, results), total=len(lyrics_texts)):
    conf_filtered = []

    for c in conf_values:
        if round(c.value,3) != 0.0:
            if c.language.iso_code_639_1 is not None:
                conf_filtered.append((c.language.name, round(c.value,3)))

    detected_langs = [lang for lang, score in conf_filtered]

    detected_languages.append(", ".join(detected_langs) if detected_langs else None)
    num_languages.append(len(detected_langs))
    language_confidences.append(", ".join([f"{lang}: {score}" for lang, score in conf_filtered]) if conf_filtered else None)


df["detected_languages"] = detected_languages
df["num_languages"] = num_languages
df["language_confidences"] = language_confidences

#find the rows where detected language is only YORUBA and change the detected language and language confidence to HINDI
for index, row in df.iterrows():
    if row["detected_languages"] == "YORUBA":
        df.at[index, "detected_languages"] = "HINDI"
        df.at[index, "language_confidences"] = "HINDI: 1.0"



df.to_csv(output_csv, index=False)
print(f"Saved enriched results")

Saved enriched results


In [ ]:
language_df = pd.read_csv("data/features-annotated-artistless-with-lyrics-and-languages.csv")

#find all the rows where there is more than one language detected
multiple_languages_df = language_df[language_df["num_languages"] > 1]

#print the ids of the rows with more than one language detected from the 24th row onwards
for index, row in multiple_languages_df.iloc[23:].iterrows():
    #print just the id
    print(row["id"])
#print the number of rows with more than one language detected
print(f"Number of songs with multiple languages detected: {len(multiple_languages_df)}")

#change the detected language to english for this id '3Z5taVHYljiWv6itioRRRw' and the language confidence to 'ENGLISH: 1.0' and num_languages to 1
language_df.loc[language_df["id"] == '3Z5taVHYljiWv6itioRRRw', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '3Z5taVHYljiWv6itioRRRw', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '3Z5taVHYljiWv6itioRRRw', "num_languages"] = 1

language_df.loc[language_df["id"] == '1k0QMCm5kF3CBjCQ37vDLl', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '1k0QMCm5kF3CBjCQ37vDLl', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '1k0QMCm5kF3CBjCQ37vDLl', "num_languages"] = 1

language_df.loc[language_df["id"] == '1pKYYY0dkg23sQQXi0Q5zN', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '1pKYYY0dkg23sQQXi0Q5zN', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '1pKYYY0dkg23sQQXi0Q5zN', "num_languages"] = 1

language_df.loc[language_df["id"] == '5xbrhx2tMMcN68IT3cJbhS', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '5xbrhx2tMMcN68IT3cJbhS', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '5xbrhx2tMMcN68IT3cJbhS', "num_languages"] = 1

language_df.loc[language_df["id"] == '7sBgFMWqDWQgrsNSpIBoe8', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '7sBgFMWqDWQgrsNSpIBoe8', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '7sBgFMWqDWQgrsNSpIBoe8', "num_languages"] = 1

language_df.loc[language_df["id"] == '4IIuCotvqijraSdnVLaFnM', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '4IIuCotvqijraSdnVLaFnM', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '4IIuCotvqijraSdnVLaFnM', "num_languages"] = 1

language_df.loc[language_df["id"] == '5W3cjX2J3tjhG8zb6u0qHn', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '5W3cjX2J3tjhG8zb6u0qHn', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '5W3cjX2J3tjhG8zb6u0qHn', "num_languages"] = 1

language_df.loc[language_df["id"] == '2prnn41CblB8B4yWACDljP', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '2prnn41CblB8B4yWACDljP', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '2prnn41CblB8B4yWACDljP', "num_languages"] = 1

language_df.loc[language_df["id"] == '0v2LUO8oTHnkjQ8MmKuXyk', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '0v2LUO8oTHnkjQ8MmKuXyk', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '0v2LUO8oTHnkjQ8MmKuXyk', "num_languages"] = 1

language_df.loc[language_df["id"] == '48C0O5CXfQdfjUCUhOs1YP', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '48C0O5CXfQdfjUCUhOs1YP', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '48C0O5CXfQdfjUCUhOs1YP', "num_languages"] = 1

language_df.loc[language_df["id"] == '665AK0ovB11McZRwfa852G', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '665AK0ovB11McZRwfa852G', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '665AK0ovB11McZRwfa852G', "num_languages"] = 1

language_df.loc[language_df["id"] == '6potEImiklXkwD9qFzpu15', "detected_languages"] = "ENGLISH"
language_df.loc[language_df["id"] == '6potEImiklXkwD9qFzpu15', "language_confidences"] = "ENGLISH: 1.0"
language_df.loc[language_df["id"] == '6potEImiklXkwD9qFzpu15', "num_languages"] = 1

#change the detected language to spanish for this id '7taXf5odg9xCAZERYfyOkS' and the language confidence to 'SPANISH: 1.0' and num_languages to 1
language_df.loc[language_df["id"] == '7taXf5odg9xCAZERYfyOkS', "detected_languages"] = "SPANISH"
language_df.loc[language_df["id"] == '7taXf5odg9xCAZERYfyOkS', "language_confidences"] = "SPANISH: 1.0"
language_df.loc[language_df["id"] == '7taXf5odg9xCAZERYfyOkS', "num_languages"] = 1

language_df.loc[language_df["id"] == '5Pg4qYTcvCUBBq3jBhKq0t', "detected_languages"] = "SPANISH"
language_df.loc[language_df["id"] == '5Pg4qYTcvCUBBq3jBhKq0t', "language_confidences"] = "SPANISH: 1.0"
language_df.loc[language_df["id"] == '5Pg4qYTcvCUBBq3jBhKq0t', "num_languages"] = 1

language_df.loc[language_df["id"] == '3VlW71ZRJPEjd17W0bW28D', "detected_languages"] = "SPANISH"
language_df.loc[language_df["id"] == '3VlW71ZRJPEjd17W0bW28D', "language_confidences"] = "SPANISH: 1.0"
language_df.loc[language_df["id"] == '3VlW71ZRJPEjd17W0bW28D', "num_languages"] = 1

language_df.loc[language_df["id"] == '3qCeq0rWK67SoBDgFd2sId', "detected_languages"] = "SPANISH"
language_df.loc[language_df["id"] == '3qCeq0rWK67SoBDgFd2sId', "language_confidences"] = "SPANISH: 1.0"
language_df.loc[language_df["id"] == '3qCeq0rWK67SoBDgFd2sId', "num_languages"] = 1

language_df.loc[language_df["id"] == '6tDDoYIxWvMLTdKpjFkc1B', "detected_languages"] = "SPANISH"
language_df.loc[language_df["id"] == '6tDDoYIxWvMLTdKpjFkc1B', "language_confidences"] = "SPANISH: 1.0"
language_df.loc[language_df["id"] == '6tDDoYIxWvMLTdKpjFkc1B', "num_languages"] = 1

language_df.loc[language_df["id"] == '7KhHA3mzt7A3LFlOJEAKpI', "detected_languages"] = "SPANISH"
language_df.loc[language_df["id"] == '7KhHA3mzt7A3LFlOJEAKpI', "language_confidences"] = "SPANISH: 1.0"
language_df.loc[language_df["id"] == '7KhHA3mzt7A3LFlOJEAKpI', "num_languages"] = 1

#change the detected language to indonesian for this id '26cvTWJq2E1QqN4jyH2OTU' and the language confidence to 'INDONESIAN: 1.0' and num_languages to 1
language_df.loc[language_df["id"] == '26cvTWJq2E1QqN4jyH2OTU', "detected_languages"] = "INDONESIAN"
language_df.loc[language_df["id"] == '26cvTWJq2E1QqN4jyH2OTU', "language_confidences"] = "INDONESIAN: 1.0"
language_df.loc[language_df["id"] == '26cvTWJq2E1QqN4jyH2OTU', "num_languages"] = 1

language_df.loc[language_df["id"] == '6Pgkp4qUoTmJIPn7ReaGxL', "detected_languages"] = "INDONESIAN"
language_df.loc[language_df["id"] == '6Pgkp4qUoTmJIPn7ReaGxL', "language_confidences"] = "INDONESIAN: 1.0"
language_df.loc[language_df["id"] == '6Pgkp4qUoTmJIPn7ReaGxL', "num_languages"] = 1

language_df.loc[language_df["id"] == '6WbADFqMvR8N5u0BvtsWQE', "detected_languages"] = "INDONESIAN"
language_df.loc[language_df["id"] == '6WbADFqMvR8N5u0BvtsWQE', "language_confidences"] = "INDONESIAN: 1.0"
language_df.loc[language_df["id"] == '6WbADFqMvR8N5u0BvtsWQE', "num_languages"] = 1

language_df.loc[language_df["id"] == '6dXiWwFrcGieqnoLYzPNp5', "detected_languages"] = "INDONESIAN"
language_df.loc[language_df["id"] == '6dXiWwFrcGieqnoLYzPNp5', "language_confidences"] = "INDONESIAN: 1.0"
language_df.loc[language_df["id"] == '6dXiWwFrcGieqnoLYzPNp5', "num_languages"] = 1

language_df.loc[language_df["id"] == '2hHeGD57S0BcopfVcmehdl', "detected_languages"] = "INDONESIAN"
language_df.loc[language_df["id"] == '2hHeGD57S0BcopfVcmehdl', "language_confidences"] = "INDONESIAN: 1.0"
language_df.loc[language_df["id"] == '2hHeGD57S0BcopfVcmehdl', "num_languages"] = 1

language_df.loc[language_df["id"] == '2u6Jm2klS4yvAlbSHlxUwI', "detected_languages"] = "INDONESIAN"
language_df.loc[language_df["id"] == '2u6Jm2klS4yvAlbSHlxUwI', "language_confidences"] = "INDONESIAN: 1.0"
language_df.loc[language_df["id"] == '2u6Jm2klS4yvAlbSHlxUwI', "num_languages"] = 1

#for id '43mnNatwMMydJ4CxMdQVnJ', change detected language to 'Korean' and language confidence to 'KOREAN: 1.0' and num_languages to 1
language_df.loc[language_df["id"] == '43mnNatwMMydJ4CxMdQVnJ', "detected_languages"] = "KOREAN"
language_df.loc[language_df["id"] == '43mnNatwMMydJ4CxMdQVnJ', "language_confidences"] = "KOREAN: 1.0"
language_df.loc[language_df["id"] == '43mnNatwMMydJ4CxMdQVnJ', "num_languages"] = 1

language_df.loc[language_df["id"] == '3wS2dXzQw58b1e3AR8YStR', "detected_languages"] = "KOREAN"
language_df.loc[language_df["id"] == '3wS2dXzQw58b1e3AR8YStR', "language_confidences"] = "KOREAN: 1.0"
language_df.loc[language_df["id"] == '3wS2dXzQw58b1e3AR8YStR', "num_languages"] = 1

language_df.loc[language_df["id"] == '2gYj9lubBorOPIVWsTXugG', "detected_languages"] = "KOREAN"
language_df.loc[language_df["id"] == '2gYj9lubBorOPIVWsTXugG', "language_confidences"] = "KOREAN: 1.0"
language_df.loc[language_df["id"] == '2gYj9lubBorOPIVWsTXugG', "num_languages"] = 1

language_df.loc[language_df["id"] == '0Q5VnK2DYzRyfqQRJuUtvi', "detected_languages"] = "KOREAN"
language_df.loc[language_df["id"] == '0Q5VnK2DYzRyfqQRJuUtvi', "language_confidences"] = "KOREAN: 1.0"
language_df.loc[language_df["id"] == '0Q5VnK2DYzRyfqQRJuUtvi', "num_languages"] = 1

language_df.loc[language_df["id"] == '2kqnQCChvRpDRAasIwc4Rh', "detected_languages"] = "KOREAN"
language_df.loc[language_df["id"] == '2kqnQCChvRpDRAasIwc4Rh', "language_confidences"] = "KOREAN: 1.0"
language_df.loc[language_df["id"] == '2kqnQCChvRpDRAasIwc4Rh', "num_languages"] = 1

language_df.loc[language_df["id"] == '32h6yQGn4oAhZ0HRsTMX26', "detected_languages"] = "KOREAN"
language_df.loc[language_df["id"] == '32h6yQGn4oAhZ0HRsTMX26', "language_confidences"] = "KOREAN: 1.0"
language_df.loc[language_df["id"] == '32h6yQGn4oAhZ0HRsTMX26', "num_languages"] = 1

#what about this song? 4n7jnSxVLd8QioibtTDBDq

#put na for this id '116H0KvKr2Zl4RPuVBruDO' in detected languages, language confidence and num_languages
language_df.loc[language_df["id"] == '116H0KvKr2Zl4RPuVBruDO', "detected_languages"] = None
language_df.loc[language_df["id"] == '116H0KvKr2Zl4RPuVBruDO', "language_confidences"] = None
language_df.loc[language_df["id"] == '116H0KvKr2Zl 4RPuVBruDO', "num_languages"] = 0

output_csv = "data/features-annotated-artistless-with-lyrics-and-languages.csv"
#language_df.to_csv(output_csv, index=False)
print(f"Saved enriched results to:\n{output_csv}")


Saved enriched results to:
data/features-annotated-artistless-with-lyrics-and-languages.csv
